> **Author:** Brendan OConnell  
> **Year:** 2026  
> **Purpose:**   
> **Key decisions:**  
> *   
> * 

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import roc_auc_score, roc_curve

In [7]:
particle_df = pd.read_parquet("../../../data/processed/particle_labeled.parquet")

# GSR vs Non_GSR only
binary_df = particle_df[particle_df["label"].isin(["GSR", "Non_GSR"])].copy()

gsr = binary_df[binary_df["label"] == "GSR"]["o"]
non_gsr = binary_df[binary_df["label"] == "Non_GSR"]["o"]

print(f"GSR particles:     {len(gsr):>10,}")
print(f"Non_GSR particles: {len(non_gsr):>10,}")

GSR particles:      1,078,946
Non_GSR particles:  1,216,039


In [8]:
# Does oxygen show up at different rates in GSR vs Non_GSR?
presence = binary_df.groupby("label")["o"].apply(lambda x: (x > 0).mean()).rename("presence_rate")
mean_when_present = binary_df[binary_df["o"] > 0].groupby("label")["o"].mean().rename("mean_conc_when_present")
overall_mean = binary_df.groupby("label")["o"].mean().rename("overall_mean")

summary = pd.concat([presence, mean_when_present, overall_mean], axis=1)
summary.index.name = "label"
print(summary.round(4))

         presence_rate  mean_conc_when_present  overall_mean
label                                                       
GSR             0.9985                 39.5348       39.4763
Non_GSR         0.9972                 39.7258       39.6148


In [ ]:
# Presence rate bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

presence.plot(kind="bar", ax=axes[0], color=["crimson", "steelblue"], edgecolor="black", width=0.5)
axes[0].set_title("Oxygen Presence Rate by Label\n(% of particles with O > 0)")
axes[0].set_ylabel("Proportion with O > 0")
axes[0].set_ylim(0, 1)
axes[0].set_xticklabels(presence.index, rotation=0)
for bar, val in zip(axes[0].patches, presence.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f"{val:.1%}", ha="center", fontsize=11, fontweight="bold")

overall_mean.plot(kind="bar", ax=axes[1], color=["crimson", "steelblue"], edgecolor="black", width=0.5)
axes[1].set_title("Mean Oxygen Concentration by Label\n(all particles, including zeros)")
axes[1].set_ylabel("Mean O concentration")
axes[1].set_xticklabels(overall_mean.index, rotation=0)
for bar, val in zip(axes[1].patches, overall_mean.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                 f"{val:.1f}", ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("outputs/oxygen__oxygen_presence_rate_by_label_n_of_particles_with_o_0.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# KDE of oxygen concentration by label (log scale to handle skew)
# Use only O > 0 to focus on the concentration signal, not presence signal
gsr_pos = gsr[gsr > 0]
non_gsr_pos = non_gsr[non_gsr > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw scale
axes[0].hist(gsr_pos.clip(upper=gsr_pos.quantile(0.99)), bins=80, density=True,
             alpha=0.5, color="crimson", label=f"GSR (n={len(gsr_pos):,})")
axes[0].hist(non_gsr_pos.clip(upper=non_gsr_pos.quantile(0.99)), bins=80, density=True,
             alpha=0.5, color="steelblue", label=f"Non_GSR (n={len(non_gsr_pos):,})")
axes[0].set_title("O Concentration (when O > 0, clipped at 99th pct)")
axes[0].set_xlabel("O concentration")
axes[0].set_ylabel("Density")
axes[0].legend()

# Log scale
axes[1].hist(np.log1p(gsr_pos), bins=80, density=True,
             alpha=0.5, color="crimson", label="GSR")
axes[1].hist(np.log1p(non_gsr_pos), bins=80, density=True,
             alpha=0.5, color="steelblue", label="Non_GSR")
axes[1].set_title("O Concentration log(1+x) scale (when O > 0)")
axes[1].set_xlabel("log(1 + O)")
axes[1].set_ylabel("Density")
axes[1].legend()

plt.tight_layout()
plt.savefig("outputs/oxygen__o_concentration_when_o_0_clipped_at_99th_pct.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nQuantile breakdown (O > 0 particles only):")
qtiles = [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
q_df = pd.DataFrame({
    "GSR":     gsr_pos.quantile(qtiles).values,
    "Non_GSR": non_gsr_pos.quantile(qtiles).values,
}, index=[f"p{int(q*100)}" for q in qtiles])
print(q_df.round(2))

In [12]:
# Mann-Whitney U test on full distributions (includes zeros)
# Sample for speed
rng = np.random.default_rng(42)
sample_size = 200_000
gsr_sample = rng.choice(gsr.values, size=min(sample_size, len(gsr)), replace=False)
non_gsr_sample = rng.choice(non_gsr.values, size=min(sample_size, len(non_gsr)), replace=False)

u_stat, p_value = stats.mannwhitneyu(gsr_sample, non_gsr_sample, alternative="two-sided")

# Cohen's d (on full arrays, vs means)
pooled_std = np.sqrt((gsr.std()**2 + non_gsr.std()**2) / 2)
cohens_d = (gsr.mean() - non_gsr.mean()) / pooled_std

# Rank-biserial correlation (effect size for Mann-Whitney)
n1, n2 = len(gsr_sample), len(non_gsr_sample)
rank_biserial = 1 - (2 * u_stat) / (n1 * n2)

print("=== Statistical Separation: Oxygen (O) ===")
print(f"\nMann-Whitney U:       {u_stat:.3e}")
print(f"p-value:              {p_value:.3e}")
print(f"Rank-biserial r:      {rank_biserial:.4f}  (|r| < 0.1 = negligible, 0.1-0.3 = small, >0.3 = medium)")
print(f"Cohen's d:            {cohens_d:.4f}  (|d| < 0.2 = small, 0.2-0.5 = medium, >0.8 = large)")
print(f"\nGSR mean O:           {gsr.mean():.2f}")
print(f"Non_GSR mean O:       {non_gsr.mean():.2f}")
print(f"GSR median O:         {gsr.median():.2f}")
print(f"Non_GSR median O:     {non_gsr.median():.2f}")

=== Statistical Separation: Oxygen (O) ===

Mann-Whitney U:       1.902e+10
p-value:              1.397e-159
Rank-biserial r:      0.0491  (|r| < 0.1 = negligible, 0.1-0.3 = small, >0.3 = medium)
Cohen's d:            -0.0094  (|d| < 0.2 = small, 0.2-0.5 = medium, >0.8 = large)

GSR mean O:           39.48
Non_GSR mean O:       39.61
GSR median O:         39.25
Non_GSR median O:     41.83


In [ ]:
# ROC AUC using oxygen alone to predict GSR (1) vs Non_GSR (0)
# Also compare presence/absence binary flag vs raw concentration
y_true = binary_df["label"].map({"GSR": 1, "Non_GSR": 0}).values
o_vals = binary_df["o"].values
o_present = (o_vals > 0).astype(float)

auc_raw = roc_auc_score(y_true, o_vals)
auc_presence = roc_auc_score(y_true, o_present)

print("=== Single-Feature ROC AUC ===")
print(f"O concentration (raw):        AUC = {auc_raw:.4f}")
print(f"O presence/absence (binary):  AUC = {auc_presence:.4f}")
print(f"\nNote: AUC=0.5 → no signal, AUC=1.0 → perfect separation")
print(f"      AUC < 0.5 means O is negatively associated with GSR")

# Plot ROC curve for raw O
fpr, tpr, _ = roc_curve(y_true, o_vals)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color="darkorange", lw=2, label=f"O concentration (AUC = {auc_raw:.4f})")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve: Oxygen as a Single Predictor of GSR")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("outputs/oxygen__roc_curve_oxygen_as_a_single_predictor_of_gsr.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Mean O concentration and presence rate per particle class
# Sorted by mean O to see if GSR-specific classes tend to have less oxygen
class_stats = binary_df.groupby(["final_class", "label"]).agg(
    count=("o", "size"),
    presence_rate=("o", lambda x: (x > 0).mean()),
    mean_o=("o", "mean"),
    median_o=("o", "median"),
).reset_index()

# Top classes by count
top_classes = binary_df["final_class"].value_counts().head(15).index
class_stats_top = class_stats[class_stats["final_class"].isin(top_classes)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Presence rate
pivot_presence = class_stats_top.pivot(index="final_class", columns="label", values="presence_rate")
pivot_presence = pivot_presence.reindex(
    pivot_presence.mean(axis=1).sort_values().index
)
pivot_presence.plot(kind="barh", ax=axes[0], color={"GSR": "crimson", "Non_GSR": "steelblue"}, alpha=0.8)
axes[0].set_title("O Presence Rate by Particle Class (top 15)")
axes[0].set_xlabel("Proportion with O > 0")
axes[0].axvline(x=0.5, color="black", linestyle="--", alpha=0.4)
axes[0].legend(loc="lower right")

# Mean O concentration
pivot_mean = class_stats_top.pivot(index="final_class", columns="label", values="mean_o")
pivot_mean = pivot_mean.reindex(pivot_presence.index)
pivot_mean.plot(kind="barh", ax=axes[1], color={"GSR": "crimson", "Non_GSR": "steelblue"}, alpha=0.8)
axes[1].set_title("Mean O Concentration by Particle Class (top 15)")
axes[1].set_xlabel("Mean O")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.savefig("outputs/oxygen__o_presence_rate_by_particle_class_top_15.png", dpi=150, bbox_inches="tight")
plt.show()

In [15]:
# Does having oxygen at all discriminate GSR from Non_GSR?
# Among O-present particles, does the AMOUNT of oxygen matter?

o_present_df = binary_df[binary_df["o"] > 0].copy()
o_absent_df  = binary_df[binary_df["o"] == 0].copy()

def gsr_rate(df):
    return (df["label"] == "GSR").mean()

print("=== Conditional Label Rates ===")
print(f"All particles:              GSR rate = {gsr_rate(binary_df):.3f}")
print(f"O > 0 (oxygen present):     GSR rate = {gsr_rate(o_present_df):.3f}")
print(f"O = 0 (no oxygen):          GSR rate = {gsr_rate(o_absent_df):.3f}")
print(f"\nTotal particles:        {len(binary_df):,}")
print(f"  O > 0:                {len(o_present_df):,}  ({len(o_present_df)/len(binary_df):.1%})")
print(f"  O = 0:                {len(o_absent_df):,}  ({len(o_absent_df)/len(binary_df):.1%})")

# Among O-present particles: AUC of O concentration alone
y_pos = o_present_df["label"].map({"GSR": 1, "Non_GSR": 0}).values
auc_pos = roc_auc_score(y_pos, o_present_df["o"].values)
print(f"\nAmong O-present particles, AUC of O concentration = {auc_pos:.4f}")

=== Conditional Label Rates ===
All particles:              GSR rate = 0.470
O > 0 (oxygen present):     GSR rate = 0.470
O = 0 (no oxygen):          GSR rate = 0.320

Total particles:        2,294,985
  O > 0:                2,289,991  (99.8%)
  O = 0:                4,994  (0.2%)

Among O-present particles, AUC of O concentration = 0.4752


In [ ]:
# Examine whether O displaces or competes with the 3 core GSR elements
# A negative relationship between O and Pb/Ba/Sb would suggest O is "anti-GSR" signal

# Sample for scatter plots
sample_idx = rng.choice(len(o_present_df), size=min(30_000, len(o_present_df)), replace=False)
plot_df = o_present_df.iloc[sample_idx].copy()
colors_map = plot_df["label"].map({"GSR": "crimson", "Non_GSR": "steelblue"})

gsr_markers = ["pb", "ba", "sb"]
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, marker in zip(axes, gsr_markers):
    ax.scatter(
        np.log1p(plot_df[marker]),
        np.log1p(plot_df["o"]),
        c=colors_map,
        alpha=0.05,
        s=2,
    )
    # Correlation per label
    for label, color in [("GSR", "darkred"), ("Non_GSR", "darkblue")]:
        sub = plot_df[plot_df["label"] == label]
        r, p = stats.spearmanr(sub[marker], sub["o"])
        ax.set_title(f"O vs {marker.upper()}\nGSR r={r:.3f}  Non_GSR r={stats.spearmanr(plot_df[plot_df['label']=='Non_GSR'][marker], plot_df[plot_df['label']=='Non_GSR']['o'])[0]:.3f}",
                     fontsize=9)
    ax.set_xlabel(f"log(1 + {marker.upper()})")
    ax.set_ylabel("log(1 + O)")

# Legend
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor="crimson", markersize=8, label="GSR"),
           Line2D([0], [0], marker="o", color="w", markerfacecolor="steelblue", markersize=8, label="Non_GSR")]
fig.legend(handles=handles, loc="upper right", fontsize=10)

plt.suptitle("Oxygen vs Core GSR Markers (O > 0 particles only, log scale)", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("outputs/oxygen__oxygen_vs_core_gsr_markers_o_0_particles_only_log_scale.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Does GSR rate change monotonically with O concentration?
# If GSR rate is flat across O buckets → O carries no marginal concentration signal
o_nonzero = binary_df[binary_df["o"] > 0].copy()
o_nonzero["o_decile"] = pd.qcut(o_nonzero["o"], q=10, labels=False, duplicates="drop")

bucket_stats = o_nonzero.groupby("o_decile").agg(
    n=("o", "size"),
    gsr_rate=("label", lambda x: (x == "GSR").mean()),
    o_min=("o", "min"),
    o_max=("o", "max"),
).reset_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(bucket_stats["o_decile"], bucket_stats["gsr_rate"],
       color="steelblue", edgecolor="black", alpha=0.8)
ax.axhline(y=gsr_rate(binary_df), color="red", linestyle="--", label=f"Overall GSR rate ({gsr_rate(binary_df):.3f})")
ax.set_title("GSR Rate Across O Concentration Deciles (O > 0 particles)")
ax.set_xlabel("O Concentration Decile (0 = lowest, 9 = highest)")
ax.set_ylabel("GSR Rate")
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.savefig("outputs/oxygen__gsr_rate_across_o_concentration_deciles_o_0_particles.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nBucket detail:")
print(bucket_stats[["o_decile", "n", "gsr_rate", "o_min", "o_max"]].to_string(index=False))

## Summary & Conclusion

OXYGEN EDA

Does oxygen add discriminative signal for GSR vs Non-GSR? Does it dilute signal?

1. PRESENCE RATE
   - Oxygen is the most prevalent element in the dataset (appears in a large
     fraction of all particles).
   - Check the presence rates above: if both GSR and Non_GSR have similar
     presence rates (~80-90%+), oxygen's presence/absence carries little signal.

2. CONCENTRATION
   - GSR particles tend to have LOWER mean oxygen than Non_GSR (driven by the
     negative correlations with GSR markers: ba-O=-0.36, sb-O=-0.30, cu-O=-0.29
     observed in the particle_eda notebook).
   - This is chemically plausible: GSR particles are dense Pb/Ba/Sb metallic
     spheres with less surface oxidation, while environmental particles (silicates,
     carbonates) are inherently oxygenated.

3. DISCRIMINATIVE POWER
   - ROC AUC near 0.5 → oxygen alone is a poor discriminator.
   - The direction of any AUC < 0.5 would mean higher O is anti-correlated
     with GSR — which makes sense physically.

4. CONCENTRATION BUCKET ANALYSIS
   - If the GSR rate is roughly flat across O deciles, concentration adds
     little on top of presence/absence.
   - Any monotonic decline in GSR rate as O increases confirms the "O = non-GSR"
     narrative.

Conclusion:
   - Keep oxygen as a feature. Even weak negative signal is informative for
     a tree-based model (XGBoost can exploit it).
   - Do not rely on oxygen alone. Its value comes from interactions (e.g.,
     high-O + low-Pb/Ba/Sb strongly suggests Non_GSR background particle).
   - Removing O risks throwing away a weak but real signal; keeping it adds
     at most marginal noise for a tree model that can learn its low importance.
   - Consider circumstances where oxygen may be diluting signal.
     Compare/contrast how those features are different when oxygen is removed.